# Pose Extraction + Annotation

In [1]:
# Copyright (c) CIIS-Lab. All rights reserved.
import os.path as osp
import os
from pathlib import Path

import copy as cp
import tempfile

import cv2
import mmcv
import mmengine
import numpy as np
import torch

from mmaction.apis import (detection_inference,
                           # inference_recognizer, init_recognizer,
                           pose_inference)
from mmaction.registry import VISUALIZERS
from mmaction.utils import frame_extract

import moviepy.editor as mpy

In [2]:
FONTFACE = cv2.FONT_HERSHEY_DUPLEX
FONTSCALE = 1

THICKNESS = 1  # int
LINETYPE = 1

In [3]:
def hex2color(h):
    """Convert the 6-digit hex string to tuple of 3 int value (RGB)"""
    return (int(h[:2], 16), int(h[2:4], 16), int(h[4:], 16))

PLATEBLUE = '03045e-023e8a-0077b6-0096c7-00b4d8-48cae4'
PLATEBLUE = PLATEBLUE.split('-')
PLATEBLUE = [hex2color(h) for h in PLATEBLUE]


def visualize(pose_config,
              frames,
              annotations,
              pose_data_samples,
              action_result,
              plate=PLATEBLUE,
              max_num=5):
    """Visualize frames with predicted annotations.

    Args:
        frames (list[np.ndarray]): Frames for visualization, note that
            len(frames) % len(annotations) should be 0.
        annotations (list[list[tuple]]): The predicted spatio-temporal
            detection results.
        pose_data_samples (list[list[PoseDataSample]): The pose results.
        action_result (str): The predicted action recognition results.
        pose_model (nn.Module): The constructed pose model.
        plate (str): The plate used for visualization. Default: PLATEBLUE.
        max_num (int): Max number of labels to visualize for a person box.
            Default: 5.

    Returns:
        list[np.ndarray]: Visualized frames.
    """

    assert max_num + 1 <= len(plate)
    frames_ = cp.deepcopy(frames)
    frames_ = [mmcv.imconvert(f, 'bgr', 'rgb') for f in frames_]
    nf, na = len(frames), len(annotations)
    assert nf % na == 0
    nfpa = len(frames) // len(annotations)
    anno = None
    h, w, _ = frames[0].shape
    scale_ratio = np.array([w, h, w, h])

    # add pose results
    if pose_data_samples:
        pose_config = mmengine.Config.fromfile(pose_config)
        visualizer = VISUALIZERS.build(pose_config.visualizer | {'line_width':5, 'bbox_color':(101,193,255), 'radius': 8})  # https://mmpose.readthedocs.io/en/latest/api.html#mmpose.visualization.PoseLocalVisualizer
        visualizer.set_dataset_meta(pose_data_samples[0].dataset_meta)
        for i, (d, f) in enumerate(zip(pose_data_samples, frames_)):
            visualizer.add_datasample(
                'result',
                f,
                data_sample=d,
                draw_gt=False,
                draw_heatmap=False,
                draw_bbox=True,
                draw_pred=True,
                show=False,
                wait_time=0,
                out_file=None,
                kpt_thr=0.3)
            frames_[i] = visualizer.get_image()

    for i in range(na):
        anno = annotations[i]
        if anno is None:
            continue
        for j in range(nfpa):
            ind = i * nfpa + j
            frame = frames_[ind]

            # add spatio-temporal action detection results
            for ann in anno:
                box = ann[0]
                label = ann[1]
                if not len(label):
                    continue
                score = ann[2]
                box = (box * scale_ratio).astype(np.int64)
                st, ed = tuple(box[:2]), tuple(box[2:])
                if not pose_data_samples:
                    cv2.rectangle(frame, st, ed, plate[0], 2)

                for k, lb in enumerate(label):
                    if k >= max_num:
                        break
                    text = abbrev(lb)
                    text = ': '.join([text, f'{score[k]:.3f}'])
                    location = (0 + st[0], 18 + k * 18 + st[1])
                    textsize = cv2.getTextSize(text, FONTFACE, FONTSCALE,
                                               THICKNESS)[0]
                    textwidth = textsize[0]
                    diag0 = (location[0] + textwidth, location[1] - 14)
                    diag1 = (location[0], location[1] + 2)
                    cv2.rectangle(frame, diag0, diag1, plate[k + 1], -1)
                    FONTCOLOR = (255, 0, 0)
                    cv2.putText(frame, text, location, FONTFACE, FONTSCALE,
                                FONTCOLOR, THICKNESS, LINETYPE)

    return frames_

In [4]:
# video_folder_path = osp.abspath("/media/ciis/680d5156-2214-4b41-baf0-6bb993bff707/ciis-compnew/Documents/ActionTracking/video/")
video_folder_path = "../assets/video/25_4_v2"
for file in os.listdir(video_folder_path):
    print(file)

mergedfile_3s_1.mp4
VID-20250416-WA0010.mp4
VID-20250416-WA0003.mp4
mergedfile_3s.mp4
VID-20250416-WA0002.mp4
merge.txt
mergedfile_2.mp4
VID-20250416-WA0006.mp4
mergedfile_1s.mp4
VID-20250416-WA0008.mp4
VID-20250416-WA0001.mp4
VID-20250416-WA0004.mp4
mergedfile_3.mp4
mergedfile.mp4
VID-20250416-WA0007.mp4
UJI
VID-20250416-WA0005.mp4
mergedfile_3s_2.mp4
mergedfile_2s.mp4
mergedfile_1.mp4


In [5]:
vidio_path = '/mergedfile_1s'
video = str(video_folder_path) + str(vidio_path) + '.mp4'
num_ver = 2

path_extractPose = f'../dataset/2025/2025_{num_ver}/extracted_pose'
path_annData = f'../dataset/2025/2025_{num_ver}/annotated_data'
Path(path_extractPose).mkdir(parents=True, exist_ok=True)
Path(path_annData).mkdir(parents=True, exist_ok=True)
    

ann_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '.csv'
pkl_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '.pkl'
out_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '_gt.mp4'

# human detection config
det_config = "../mmaction2/demo/demo_configs/faster-rcnn_r50_fpn_2x_coco_infer.py"
det_checkpoint = 'http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth'
det_score_thr = 0.9
#det_cat_id = 0

# pose estimation config
pose_config = '../mmaction2/demo/demo_configs/td-hm_hrnet-w32_8xb64-210e_coco-256x192_infer.py'
pose_checkpoint = 'https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth'

# use skeleton-based method
use_skeleton_stdet = True
use_skeleton_recog = True

# skeleton-based spatio-temporal action classification config
label_map_stdet = "../mmaction2/tools/data/ciis/ciis_label_map.txt"

predict_stepsize = 16  # must even int, give out a spatio-temporal detection prediction per n frames
output_stepsize = 1  # show one frame per n frames in the demo, we should have: predict_stepsize % output_stepsize == 0, speedUp/slowDown video output
output_fps = 20  # the fps of demo video output, will speedUp/slowDown video output, must equal to (video_input_fps/output_stepsize) to get normal speed

device = 'cuda:0'

In [18]:
# def load_label_map(file_path):
#     """Load Label Map.

#     Args:
#         file_path (str): The file path of label map.

#     Returns:
#         dict: The label map (int -> label name).
#     """
#     lines = open(file_path).readlines()
#     lines = [x.strip().split(': ') for x in lines]
#     return {int(x[0]): x[1] for x in lines}


def abbrev(name):
    """Get the abbreviation of label name:

    'take (an object) from (a person)' -> 'take ... from ...'
    """
    while name.find('(') != -1:
        st, ed = name.find('('), name.find(')')
        name = name[:st] + '...' + name[ed + 1:]
    return name

def pack_result(human_detection, result, img_h, img_w):
    """Short summary.

    Args:
        human_detection (np.ndarray): Human detection result.
        result (type): The predicted label of each human proposal.
        img_h (int): The image height.
        img_w (int): The image width.

    Returns:
        tuple: Tuple of human proposal, label name and label score.
    """
    human_detection[:, 0::2] /= img_w
    human_detection[:, 1::2] /= img_h
    results = []
    if result is None:
        return None
    for prop, res in zip(human_detection, result):
        res.sort(key=lambda x: -x[1])
        results.append(
            (prop.data.cpu().numpy(), [x[0] for x in res], [x[1]
                                                            for x in res]))
    return results


def expand_bbox(bbox, h, w, ratio=1.25):
    x1, y1, x2, y2 = bbox
    center_x = (x1 + x2) // 2
    center_y = (y1 + y2) // 2
    width = x2 - x1
    height = y2 - y1

    square_l = max(width, height)
    new_width = new_height = square_l * ratio

    new_x1 = max(0, int(center_x - new_width / 2))
    new_x2 = min(int(center_x + new_width / 2), w)
    new_y1 = max(0, int(center_y - new_height / 2))
    new_y2 = min(int(center_y + new_height / 2), h)
    return (new_x1, new_y1, new_x2, new_y2)


def cal_iou(box1, box2):
    xmin1, ymin1, xmax1, ymax1 = box1
    xmin2, ymin2, xmax2, ymax2 = box2

    s1 = (xmax1 - xmin1) * (ymax1 - ymin1)
    s2 = (xmax2 - xmin2) * (ymax2 - ymin2)

    xmin = max(xmin1, xmin2)
    ymin = max(ymin1, ymin2)
    xmax = min(xmax1, xmax2)
    ymax = min(ymax1, ymax2)

    w = max(0, xmax - xmin)
    h = max(0, ymax - ymin)
    intersect = w * h
    union = s1 + s2 - intersect
    iou = intersect / union

    return iou


# clip_pose_extraction
def skeleton_based_stdet(predict_stepsize, video,
                         # skeleton_config, skeleton_stdet_checkpoint, device, action_score_thr, label_map,
                         human_detections, pose_results, num_frame, clip_len, frame_interval, h, w):
    window_size = clip_len * frame_interval
    assert clip_len % 2 == 0, 'We would like to have an even clip_len'
    timestamps = np.arange(window_size // 2, num_frame + 1 - window_size // 2,
                           predict_stepsize)

    # skeleton_config = mmengine.Config.fromfile(skeleton_config)
    # num_class = max(label_map.keys()) + 1  # for AVA dataset (81)
    # skeleton_config.model.cls_head.num_classes = num_class
    # skeleton_stdet_model = init_recognizer(skeleton_config,
    #                                        skeleton_stdet_checkpoint,
    #                                        device)

    skeleton_predictions = []
    skeleton_datasets = []

    print('Building skeleton datasets from existing keypoint data for each clip')
    prog_bar = mmengine.ProgressBar(len(timestamps))
    for timestamp in timestamps:  # iterate each clip
        proposal = human_detections[timestamp - 1] # get bboxes for persons in timestamp (first frame of clip)
        if proposal.shape[0] == 0:  # no people detected
            skeleton_predictions.append(None)
            continue

        start_frame = timestamp - (clip_len // 2 - 1) * frame_interval
        frame_inds = start_frame + np.arange(0, window_size, frame_interval)
        frame_inds = list(frame_inds - 1)
        num_frame = len(frame_inds)  # 30

        pose_result = [pose_results[ind] for ind in frame_inds]  # grouping frames poses for each clip

        skeleton_prediction = []
        for i in range(proposal.shape[0]):  # num_person  # iterate each bbox in timestamp (first frame of clip)
            skeleton_prediction.append([])

            fake_anno = dict(
                frame_dir=osp.splitext(osp.basename(video))[0]+"_"+str(timestamp+(i+1)*0.001),
                label=-1,
                img_shape=(h, w),
                original_shape=(h, w),
                num_clips=1,
                total_frames=num_frame
            )
            num_person = 1

            num_keypoint = 17
            keypoint = np.zeros(
                (num_person, num_frame, num_keypoint, 2))  # M T V 2
            keypoint_score = np.zeros(
                (num_person, num_frame, num_keypoint))  # M T V

            # pose matching
            person_bbox = proposal[i][:4]  # get bbox for a person in timestamp (first frame of clip)
            area = expand_bbox(person_bbox, h, w)  # bbox expanded by 1.25 ratio with square shape

            for j, poses in enumerate(pose_result):  # num_frame  # iterate each frame of clip
                max_iou = float('-inf')
                index = -1
                if len(poses['keypoints']) == 0:
                    continue
                for k, bbox in enumerate(poses['bboxes']):  # iterate each bbox/pose in each frame
                    iou = cal_iou(bbox, area)  # compare each bbox in each frame with current area (calculate_intersect/union)
                    if max_iou < iou:
                        index = k  # pose from the biggest intersect/union (iou) will be considered
                        max_iou = iou
                keypoint[0, j] = poses['keypoints'][index]
                keypoint_score[0, j] = poses['keypoint_scores'][index]

            fake_anno['keypoint'] = keypoint
            fake_anno['keypoint_score'] = keypoint_score

            skeleton_datasets.append(fake_anno)
            # output = inference_recognizer(skeleton_stdet_model, fake_anno)
            # # for multi-label recognition
            # score = output.pred_score.tolist()
            # for k in range(len(score)):  # 81
            #     if k not in label_map:
            #         continue
            #     if score[k] > action_score_thr:
            #         skeleton_prediction[i].append((label_map[k], score[k]))
            skeleton_prediction[i].append(("annotate!", timestamp + (i+1)*0.001))

        skeleton_predictions.append(skeleton_prediction)
        prog_bar.update()

    return timestamps, skeleton_predictions, skeleton_datasets

In [19]:
print(video)

../assets/video/25_4_v2/mergedfile_3s_2.mp4


In [20]:
#args = parse_args()
tmp_dir = tempfile.TemporaryDirectory()
frame_paths, original_frames = frame_extract(
    video_path=video, 
    # 720,
    out_dir=tmp_dir.name)
num_frame = len(frame_paths)
h, w, _ = original_frames[0].shape

In [21]:
print(original_frames[0].shape)

(720, 1280, 3)


In [22]:
# get Human detection results
human_detections, _ = detection_inference(
    det_config,
    det_checkpoint,
    frame_paths,
    det_score_thr,
    device=device)
torch.cuda.empty_cache()

# get Pose estimation results
pose_datasample = None
pose_results, pose_datasample = pose_inference(
    pose_config,
    pose_checkpoint,
    frame_paths,
    human_detections,
    device=device)
torch.cuda.empty_cache()

04/24 19:55:37 - mmengine - WARNING - The current default scope "mmpose" is not "mmdet", `init_default_scope` will force set the currentdefault scope to "mmdet".


Loads checkpoint by http backend from path: http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth
Performing Human Detection for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>] 2644/2644, 23.2 task/s, elapsed: 114s, ETA:     0s
04/24 19:57:31 - mmengine - WARNING - The current default scope "mmdet" is not "mmpose", `init_default_scope` will force set the currentdefault scope to "mmpose".
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing Human Pose Estimation for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>>>] 2644/2644, 13.6 task/s, elapsed: 195s, ETA:     0s


In [23]:
print(len(human_detections))
human_detections

2644


[array([[ 574.3008 ,  109.35697,  705.31366,  425.40826],
        [ 864.7462 ,  437.63824, 1162.3641 ,  707.36017]], dtype=float32),
 array([[ 570.283  ,  110.92311,  706.57837,  426.70352],
        [ 872.00433,  439.05844, 1161.0083 ,  706.0947 ]], dtype=float32),
 array([[ 566.716  ,  107.78191,  699.2989 ,  425.23898],
        [ 872.88556,  440.7049 , 1161.8137 ,  713.2746 ]], dtype=float32),
 array([[ 567.2411 ,  106.14809,  697.6639 ,  427.5955 ],
        [ 872.8963 ,  437.30554, 1163.6373 ,  715.02985]], dtype=float32),
 array([[ 561.7945 ,  104.73631,  694.5846 ,  426.75403],
        [ 868.6727 ,  440.07452, 1158.767  ,  713.2044 ]], dtype=float32),
 array([[ 560.6565  ,  107.733864,  688.94336 ,  422.93002 ],
        [ 866.75934 ,  440.65414 , 1160.2625  ,  713.4853  ]],
       dtype=float32),
 array([[ 548.71796,  102.60347,  671.6561 ,  422.00623],
        [ 870.7856 ,  433.677  , 1161.668  ,  707.4993 ]], dtype=float32),
 array([[ 545.66754,  103.78081,  664.69006,  419.1287

In [24]:
stdet_preds = None

print('Use skeleton-based SpatioTemporal Action Detection')
# clip_len, frame_interval = 30, 1
clip_len, frame_interval = predict_stepsize, 1

# clip_pose_extraction
timestamps, stdet_preds, skeleton_datasets = skeleton_based_stdet(predict_stepsize, video,
                                                                  # skeleton_config,
                                                                  # skeleton_stdet_checkpoint,
                                                                  # device,
                                                                  # action_score_thr,
                                                                  # stdet_label_map,
                                                                  human_detections,
                                                                  pose_results, num_frame,
                                                                  clip_len,
                                                                  frame_interval, h, w)
for i in range(len(human_detections)):
    det = human_detections[i]
    # det[:, 0:4:2] *= w_ratio
    # det[:, 1:4:2] *= h_ratio
    det[:, 0:4:2] *= 1
    det[:, 1:4:2] *= 1
    human_detections[i] = torch.from_numpy(det[:, :4]).to(device)

Use skeleton-based SpatioTemporal Action Detection
Building skeleton datasets from existing keypoint data for each clip
[>>>>>>>>>>>>>>>>>>>>>>>>>>>> ] 163/165, 648.0 task/s, elapsed: 0s, ETA:     0s

## Annotation

In [25]:
anno = ""
for clip in stdet_preds:
    if clip == None:
        continue
    for person_attr in clip:
        anno += str(person_attr[0][0]) + "," + str(person_attr[0][1]) + "\n"

with open(ann_filename,'w') as data:
    data.write(anno)

mmengine.dump(skeleton_datasets, pkl_filename)

In [26]:
stdet_results = []
for timestamp, prediction in zip(timestamps, stdet_preds):
    human_detection = human_detections[timestamp - 1]
    stdet_results.append(
        pack_result(human_detection, prediction, h, w))

def dense_timestamps(timestamps, n):
    """Make it nx frames."""
    old_frame_interval = (timestamps[1] - timestamps[0])
    start = timestamps[0] - old_frame_interval / n * (n - 1) / 2
    new_frame_inds = np.arange(
        len(timestamps) * n) * old_frame_interval / n + start
    return new_frame_inds.astype(np.int64)

dense_n = int(predict_stepsize / output_stepsize)
output_timestamps = dense_timestamps(timestamps, dense_n) + 1
frames = [
    cv2.imread(frame_paths[timestamp - 1])
    for timestamp in output_timestamps
]

pose_datasample = [
    pose_datasample[timestamp - 1] for timestamp in output_timestamps
]

In [27]:
vis_frames = visualize(pose_config, frames, stdet_results, pose_datasample,
                       None)
vid = mpy.ImageSequenceClip(vis_frames, fps=output_fps)
vid.write_videofile(out_filename)
tmp_dir.cleanup()

Moviepy - Building video ../dataset/2025/2025_2/extracted_pose/mergedfile_3s_2_gt.mp4.
Moviepy - Writing video ../dataset/2025/2025_2/extracted_pose/mergedfile_3s_2_gt.mp4



Moviepy - Done !
Moviepy - video ready ../dataset/2025/2025_2/extracted_pose/mergedfile_3s_2_gt.mp4


In [28]:
frame_extract(
    out_filename, out_dir="../extracted/")

(['../extracted/mergedfile_3s_2_gt/img_000001.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000002.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000003.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000004.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000005.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000006.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000007.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000008.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000009.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000010.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000011.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000012.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000013.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000014.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000015.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000016.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000017.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000018.jpg',
  '../extracted/mergedfile_3s_2_gt/img_000019.jpg',
  '../extrac

# Add Label to Datasets

In [10]:
anned_filename = '../dataset/2025/2025_2/extracted_pose' + str(vidio_path) + '_edited.csv'
pkl_final = '../dataset/2025/2025_2/annotated_data' + str(vidio_path) + '.pkl'

In [7]:
import csv

def load_label_map(file_path):
    """Load Label Map.

    Args:
        file_path (str): The file path of label map.

    Returns:
        dict: The label map (label name -> int).
    """
    lines = open(file_path).readlines()
    lines = [x.strip().split(': ') for x in lines]
    return {x[1]: int(x[0]) for x in lines}

In [8]:
stdet_label_map = load_label_map(label_map_stdet)

stdet_label_map

{'berdiri': 0,
 'berjalan': 1,
 'berjongkok': 2,
 'merayap': 3,
 'melempar': 4,
 'membidik senapan': 5,
 'membidik pistol': 6,
 'memukul': 7,
 'menendang': 8,
 'menusuk': 9}

In [15]:
custom_annos = []

with open(anned_filename, newline='') as csvfile:
    spamreader = csv.reader(csvfile, delimiter=',')
    for row in spamreader:
        if row[0] == 'none':
            continue
        label = stdet_label_map[row[0]]
        id = float(row[1])
        custom_annos.append([id, label])

In [16]:
skeleton_datasets = mmengine.load(pkl_filename)

custom_dataset = []

In [17]:
for index, ann in enumerate(custom_annos):

    fake_anno = dict(
        frame_dir=osp.splitext(osp.basename(pkl_filename))[0]+"_"+str(ann[0]))

    for j, data in enumerate(skeleton_datasets):
        if fake_anno['frame_dir'] == data['frame_dir']:
            fake_anno['frame_dir'] += '_' + str(index)
            fake_anno['label'] = ann[1]
            fake_anno['img_shape'] = data['img_shape']
            fake_anno['original_shape'] = data['original_shape']
            fake_anno['num_clips'] = 1
            fake_anno['total_frames'] = data['total_frames']
            fake_anno['clip_len'] = data['total_frames']
            fake_anno['keypoint'] = data['keypoint']
            fake_anno['keypoint_score'] = data['keypoint_score']

    custom_dataset.append(fake_anno)

In [18]:
mmengine.dump(custom_dataset, pkl_final)

# Visualize

In [23]:
import csv

def visualize(pose_config,
              frames,
              annotations,
              pose_data_samples,
              action_result,
              plate=PLATEBLUE,
              max_num=5):
    """Visualize frames with predicted annotations.

    Args:
        frames (list[np.ndarray]): Frames for visualization, note that
            len(frames) % len(annotations) should be 0.
        annotations (list[list[tuple]]): The predicted spatio-temporal
            detection results.
        pose_data_samples (list[list[PoseDataSample]): The pose results.
        action_result (str): The predicted action recognition results.
        pose_model (nn.Module): The constructed pose model.
        plate (str): The plate used for visualization. Default: PLATEBLUE.
        max_num (int): Max number of labels to visualize for a person box.
            Default: 5.

    Returns:
        list[np.ndarray]: Visualized frames.
    """

    assert max_num + 1 <= len(plate)
    frames_ = cp.deepcopy(frames)
    frames_ = [mmcv.imconvert(f, 'bgr', 'rgb') for f in frames_]
    nf, na = len(frames), len(annotations)
    assert nf % na == 0
    nfpa = len(frames) // len(annotations)
    anno = None
    h, w, _ = frames[0].shape
    scale_ratio = np.array([w, h, w, h])

    # add pose results
    if pose_data_samples:
        pose_config = mmengine.Config.fromfile(pose_config)
        visualizer = VISUALIZERS.build(pose_config.visualizer | {'line_width':5, 'bbox_color':(101,193,255), 'radius': 8})  # https://mmpose.readthedocs.io/en/latest/api.html#mmpose.visualization.PoseLocalVisualizer
        visualizer.set_dataset_meta(pose_data_samples[0].dataset_meta)
        for i, (d, f) in enumerate(zip(pose_data_samples, frames_)):
            visualizer.add_datasample(
                'result',
                f,
                data_sample=d,
                draw_gt=False,
                draw_heatmap=False,
                draw_bbox=True,
                draw_pred=True,
                show=False,
                wait_time=0,
                out_file=None,
                kpt_thr=0.3)
            frames_[i] = visualizer.get_image()

    for i in range(na):
        anno = annotations[i]
        if anno is None:
            continue
        for j in range(nfpa):
            ind = i * nfpa + j
            frame = frames_[ind]

            # add spatio-temporal action detection results
            for ann in anno:
                box = ann[0]
                label = ann[1]
                if not len(label):
                    continue
                score = ann[2]
                box = (box * scale_ratio).astype(np.int64)
                st, ed = tuple(box[:2]), tuple(box[2:])
                if not pose_data_samples:
                    cv2.rectangle(frame, st, ed, plate[0], 2)

                for k, lb in enumerate(label):
                    if k >= max_num:
                        break
                    text = abbrev(lb)
                    # text = ': '.join([text, f'{(score[k]*100):.1f}%'])  # to add score
                    location = (0 + st[0], 18 + k * 18 + st[1])
                    textsize = cv2.getTextSize(text, FONTFACE, FONTSCALE,
                                               THICKNESS)[0]
                    textwidth = textsize[0]
                    diag0 = (location[0] + textwidth, location[1] - 14)
                    diag1 = (location[0], location[1] + 2)
                    cv2.rectangle(frame, diag0, diag1, plate[k + 1], -1)
                    bahaya = ['melempar', 'membidik senapan', 'membidik pistol', 'memukul', 'menendang', 'menusuk']
                    FONTCOLOR = (255, 0, 0) if lb in bahaya else (255, 255, 255)
                    cv2.putText(frame, text, location, FONTFACE, FONTSCALE,
                                FONTCOLOR, THICKNESS, LINETYPE)

    return frames_

In [24]:
stdet_preds_gt = list([])

# timestamps == ambil dari atas
# stdet_preds == ambil dari atas

for timestamp, stdet_pred in zip(timestamps, stdet_preds):
    timestmp_anno = list([])
    if stdet_pred != None:
        for i, object in enumerate(stdet_pred):
            object_anno = list([])
            with open(anned_filename, newline='') as csvfile:
                spamreader = csv.reader(csvfile, delimiter=',')
                for row in spamreader:
                    if row[0] == 'none':
                        continue
                    elif float(row[1]) == (float(timestamp) + (i+1)*0.001):
                        object_anno.append(tuple([row[0], np.random.uniform(0.4, 1.0)]))
            timestmp_anno.append(object_anno)
    stdet_preds_gt.append(timestmp_anno)

# stdet_preds = list[list[tuple]]
# [
#     [[(), ()],[()],[]],
#     [[],[]]
# ]

In [25]:
#args = parse_args()  ##test bug
tmp_dir = tempfile.TemporaryDirectory()
frame_paths, original_frames = frame_extract(
    video, 720, out_dir=tmp_dir.name)
num_frame = len(frame_paths)
h, w, _ = original_frames[0].shape

In [ ]:
stdet_results = []

# human_detections == ambil dari atas
# new_h, new_w == ambil dari atas
# predict_stepsize == ambil dari atas
# output_stepsize == ambil dari atas
# frame_paths == ambil dari atas
# pose_datasample == ambil dari atas
##################################### rerun the necessary cell if lost

for timestamp, prediction in zip(timestamps, stdet_preds_gt):
    human_detection = human_detections[timestamp - 1]
    stdet_results.append(
        pack_result(human_detection, prediction, h, w))

def dense_timestamps(timestamps, n):
    """Make it nx frames."""
    old_frame_interval = (timestamps[1] - timestamps[0])
    start = timestamps[0] - old_frame_interval / n * (n - 1) / 2
    new_frame_inds = np.arange(
        len(timestamps) * n) * old_frame_interval / n + start
    return new_frame_inds.astype(np.int64)

dense_n = int(predict_stepsize / output_stepsize)
# output_timestamps = dense_timestamps(timestamps, dense_n)
output_timestamps = dense_timestamps(timestamps, dense_n) + 1
frames = [
    cv2.imread(frame_paths[timestamp - 1])
    # cv2.imread("../854x480-white-solid-color-background.jpg")
    for timestamp in output_timestamps
]

pose_datasample = [
    pose_datasample[timestamp - 1] for timestamp in output_timestamps
]

In [27]:
# pose_config == ambil dari atas
# frames == ambil dari atas

# pose_datasample == ambil dari atas
out_filename_2 = '../data/skeleton/to-anno' + str(vidio_path) + '_gt2.mp4'


vis_frames = visualize(pose_config, frames, stdet_results, pose_datasample,
                       None)
vid = mpy.ImageSequenceClip(vis_frames, fps=output_fps)
vid.write_videofile(out_filename_2)
tmp_dir.cleanup()

Moviepy - Building video ../data/skeleton/to-anno/70d_1s3_gt2.mp4.
Moviepy - Writing video ../data/skeleton/to-anno/70d_1s3_gt2.mp4



Moviepy - Done !
Moviepy - video ready ../data/skeleton/to-anno/70d_1s3_gt2.mp4


### Move the pkl file to the 'dataset' folder in data/2025_1/train

# Combine PKL

In [28]:
import os
import re

In [ ]:
pickles_path = '../dataset/2025/2025_2/annotated_data'

split_ratio = 0.5
split_str = str(split_ratio).replace(".", "s")
base_dir = '../dataset/2025/train_dataset/'
pattern = f'ciis_{split_str}_v'
ext = '.pkl'

# Auto Versioning
versions = [
    int(m.group(1)) for f in os.listdir(base_dir)
    if (m := re.match(fr'{pattern}(\d+){re.escape(ext)}', f))
]
next_version = max(versions, default=0) + 1

# Final file path
combined_pkl = os.path.join(base_dir, f'{pattern}{next_version}{ext}')
print(combined_pkl)

../dataset/2025/train_dataset/ciis_0s9_v1.pkl


In [36]:
custom_datasets = dict(split=dict(xsub_train=[],
                                  xsub_val=[],
                                  xview_train=[],
                                  xview_val=[]),
                       annotations=[])

In [37]:
for file in os.listdir(pickles_path):
    print(file)
    if not file.endswith('.pkl'):
        continue
    custom_dataset = mmengine.load(os.path.join(pickles_path, file))
    for i, data in enumerate(custom_dataset):
        custom_datasets['annotations'].append(data)
        if (i % 10) < (split_ratio * 10):
            custom_datasets['split']['xsub_train'].append(data['frame_dir'])
            custom_datasets['split']['xview_train'].append(data['frame_dir'])
        else:
            custom_datasets['split']['xsub_val'].append(data['frame_dir'])
            custom_datasets['split']['xview_val'].append(data['frame_dir'])

mergedfile_1s.pkl


In [38]:
mmengine.dump(custom_datasets, combined_pkl)